In [1]:
from collections import defaultdict
from datetime import datetime
import hashlib
import json
import os

In [2]:
ORIGIN = "/home/tteuz/Desktop/mestrado/meta_pklot/datasets/organized/PKLot"

In [3]:
def get_car_id(car_id, date):
    return hashlib.sha256(f"{car_id},{','.join(map(str, date))}".encode()).hexdigest()[:10]


def get_spot_id(rotated_rect, date):
    flat = [round(x, 4) for item in rotated_rect for x in (item if isinstance(item, list) else [item])]
    combined = flat + date

    combined_string = ','.join(map(str, combined))
    return hashlib.sha256(combined_string.encode()).hexdigest()[:10]

In [4]:
def get_test_artifacts(ds_path, file):
    with open(f"{ds_path}/{file}", "r") as json_file:
        data = json.load(json_file)

    id_to_date = {img["id"]: img["date"] for img in data["images"]}
    annotations = defaultdict(list)

    for ann in data["annotations"]:
        if ann["car_id"] == -2:
            ann["category_id"] = 0
        
        date = id_to_date[ann["image_id"]]
        ann["spot_id"] = get_spot_id(ann["best_rotated_rect"], date)
        ann["car_id"] = get_car_id(ann["car_id"], date) if ann["category_id"] == 1 else -1
        annotations[ann["image_id"]].append(ann)

    for img in data["images"]:
        img["file_name"] = f"{ds_path}/{img['file_name']}"

    images = sorted(data["images"], key=lambda x: datetime(*x["date"], *x["time"]))
    return images, dict(annotations)

In [5]:
jsons = [file for file in os.listdir(ORIGIN) if ".json" in file]
jsons = sorted(jsons)

for file in jsons:
    images, annotations = get_test_artifacts(ORIGIN, file)
    subset = file.replace(".json", "").upper()

    id_to_date = {img["id"]: datetime(*img["date"]) for img in images}
    spots_per_date = defaultdict(set)

    for img_id, ann in annotations.items():
        date = id_to_date[img_id]
        spots_per_date[date].update(spot["spot_id"] for spot in ann)

    total_length = sum(len(spots) for spots in spots_per_date.values())
    combined_set = set().union(*spots_per_date.values())

    print(f"Subset: {subset} - Total spots by date (sum): {total_length}, Unique spots: {len(combined_set)} - Max sports: {max(len(spots) for spots in spots_per_date.values())} - Min spots: {min(len(spots) for spots in spots_per_date.values())}")

Subset: PUCPR_SPOTS - Total spots by date (sum): 7215, Unique spots: 7215 - Max sports: 195 - Min spots: 195
Subset: UFPR04_SPOTS - Total spots by date (sum): 1088, Unique spots: 1088 - Max sports: 37 - Min spots: 35
Subset: UFPR05_SPOTS - Total spots by date (sum): 1518, Unique spots: 1518 - Max sports: 46 - Min spots: 46
